<a href="https://colab.research.google.com/github/HoFangHuy/AI/blob/main/mong_day_la_lan_cuoi_cung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from PIL import Image
import cv2
import os

# Load mô hình YOLOv5s có sẵn từ PyTorch Hub
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Dự đoán và crop món ăn từ ảnh
def detect_and_crop(image_path, output_dir="crops"):
    os.makedirs(output_dir, exist_ok=True)

    results = model(image_path)
    detections = results.xyxy[0]  # bounding boxes

    image = Image.open(image_path).convert('RGB')
    for i, (*box, conf, cls) in enumerate(detections):
        x1, y1, x2, y2 = map(int, box)
        crop = image.crop((x1, y1, x2, y2))
        crop = crop.resize((224, 224))  # Resize về đúng input CNN
        crop.save(f"{output_dir}/crop_{i}.jpg")

    print(f"Đã crop {len(detections)} đối tượng vào thư mục `{output_dir}`")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# Thông số
img_size = (224, 224)
#batch_size = 32
data_path = '/content/drive/MyDrive/Data AI'

# Tiền xử lý dữ liệu
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_path,
    target_size=img_size,
    #batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    data_path,
    target_size=img_size,
    #batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(512, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(train_gen.num_classes, activation='softmax')
])

model.compile(optimizer=Adam(0.0005), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=10)


Found 1019 images belonging to 10 classes.
Found 249 images belonging to 10 classes.
Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 581s 18s/step - accuracy: 0.1848 - loss: 2.7488 - val_accuracy: 0.1566 - val_loss: 2.7313
Epoch 2/10
14/32 ━━━━━━━━━━━━━━━━━━━━ 2:08 7s/step - accuracy: 0.3287 - loss: 2.1244

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Danh sách món ăn và giá tiền
labels = ['Ca hu kho', 'Canh cai', 'Canh chua', 'Dau hu sot ca', 'Ga chien', 'Rau muong xao toi', 'Thit kho', 'Thit kho trung', 'Trung chien']
prices = {
    'Ca hu kho': 12000,
    'Canh cai': 7000,
    'Canh chua': 8000,
    'Dau hu sot ca': 12000,
    'Ga chien': 12000,
    'Rau muong xao toi': 3000,
    'Thit kho': 8000,
    'Thit kho trung': 8000,
    'Trung chien': 5000
}

def recognize_and_bill(crop_folder="crops"):
    total = 0
    print("=== HÓA ĐƠN MÓN ĂN ===")
    for filename in os.listdir(crop_folder):
        if filename.endswith(".jpg"):
            img_path = os.path.join(crop_folder, filename)
            img = image.load_img(img_path, target_size=(224, 224))
            img_array = image.img_to_array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            pred = model.predict(img_array)
            label_idx = np.argmax(pred)
            label = labels[label_idx]
            price = prices.get(label, 0)
            total += price
            print(f"- {label.replace('_', ' ').title()}: {price:,} VNĐ")

    print(f"Tổng cộng: {total:,} VNĐ")


In [ ]:
image_path = "test_tray.jpg"  # Đường dẫn ảnh khay cơm
detect_and_crop(image_path, output_dir="crops")
recognize_and_bill("crops")

In [ ]:
model.save('my_cnn_model.h5')